# Observe the ReAct cycle

You are on a practice security team at a company. This notebook prints simple status updates while the agent works.

![ReAct loop event stream](figures/react-loop-event-stream.svg)

The first model request asks for a tool. The tool result returns to the agent, and the second model request writes the answer.

## This lab observes the built-in cycle

This notebook does not write its own Python while loop. Calling reply_stream starts AgentScope's built-in ReAct cycle and gives the notebook each status update as it happens.

## How the cycle works

1. The agent sends the employee's question and its instructions to the model.
2. The model asks to use get_ip_details because it needs information from the practice list.
3. AgentScope runs the Python function and gives its result back to the model.
4. The model uses the result to write the answer.

One tool is enough to show the cycle: model request → tool → model request → answer. The setting max_iters=3 is a safety limit: it stops the agent if it keeps asking for tools instead of reaching an answer.

In [ ]:
from collections import Counter
import os
from dotenv import load_dotenv
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel
from agentscope.tool import FunctionTool, Toolkit

In [ ]:
def get_ip_details(ip_address: str) -> dict[str, str]:
    """Get details from the fixed practice list.

    Inputs:
        ip_address: An internet address, such as 192.0.2.44.
    Output:
        A matching record or a no-record result.
    Process:
        1. Search the fixed list using the input address.
        2. Return saved details when found.
        3. Otherwise return no record.
    """
    records = {
        "192.0.2.44": {"local_result": "suspicious", "details": "A person should look into this address in the practice scenario."},
        "198.51.100.10": {"local_result": "known benign", "details": "Expected activity in the practice scenario."},
    }
    return records.get(ip_address, {"local_result": "no record", "details": "The practice list has no details for this address."})


def describe_event(event) -> str:
    """Turn an AgentScope update into plain language.

    Inputs:
        event: One status update from the agent.
    Output:
        A short sentence explaining the update.
    Process:
        1. Read the update name.
        2. Match known names to simple sentences.
        3. Use a general sentence for other updates.
    """
    name = event.__class__.__name__
    messages = {
        "ModelCallStartEvent": "Model request starts.",
        "ToolCallStartEvent": f"Tool requested: {getattr(event, 'tool_call_name', 'unknown tool')}.",
        "ToolResultStartEvent": f"Tool starts: {getattr(event, 'tool_call_name', 'unknown tool')}.",
        "ToolResultEndEvent": "Tool result is ready.",
        "ReplyEndEvent": "Agent finished its answer.",
    }
    return messages.get(name, f"Update: {name}")

# Turn the function into a tool and put it in the agent toolbox.
toolkit = Toolkit(tools=[FunctionTool(get_ip_details, is_read_only=True)])

## Function examples

| Function | Input | Output |
| --- | --- | --- |
| get_ip_details | 192.0.2.44 | A record marked suspicious. |
| get_ip_details | 198.51.100.23 | A no-record result. |
| describe_event | a tool-start update | Tool starts: get_ip_details. |

In [ ]:
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")
if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=180),
)
agent = Agent(
    name="practice_assistant",
    system_prompt="For every internet-address question, call get_ip_details before answering. Use only the tool result.",
    model=model,
    toolkit=toolkit,
    react_config=ReActConfig(max_iters=3),
)

In [ ]:
question = Msg(name="analyst", role="user", content=[TextBlock(text="What details does the practice list have about 192.0.2.44?")])

# reply_stream starts AgentScope's built-in cycle and gives each update as it happens.
# Count only the updates that show the main cycle.
important_events = Counter()
labels = {
    "ModelCallStartEvent": "Model request starts",
    "ToolCallStartEvent": "Tool requested",
    "ToolResultStartEvent": "Tool starts",
    "ToolResultEndEvent": "Tool result is ready",
    "ReplyEndEvent": "Agent finished its answer",
}

async for item in agent.reply_stream(question, yield_final_msg=True):
    if isinstance(item, Msg):
        print("FINAL ANSWER:")
        print("".join(block.text for block in item.content if isinstance(block, TextBlock)))
    else:
        event_name = item.__class__.__name__
        if event_name in labels:
            important_events[labels[event_name]] += 1
        print(describe_event(item))

print("\nEVENT SUMMARY:")
for label in labels.values():
    print(f"{label}: {important_events[label]}")